In [32]:
import math
import pandas as pd
import tensorflow as tf
import numpy as np
import os
from itertools import chain, combinations
from sklearn.linear_model import LogisticRegression
from scipy.special import expit 
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

In [33]:
def Get_Test_Train(seller, file_path, percentage_train, frac_include):


    f = open("Data/Results/parameters.csv", "a")
    
    #Read in the data
    df_sales_data = pd.read_csv(file_path, index_col=0).reset_index(drop = True)

    #Update the item_id column so that we have consecutive integers as IDs - need this for estimating lambdas
    df_item_id = pd.DataFrame({"item_id":df_sales_data.item_id.unique()})
    df_item_id.index.name = "new_item_id"
    df_item_id.reset_index(inplace=True)
    df_sales_data = df_sales_data.merge(df_item_id, how="inner", on = "item_id").sort_values(by = "pvid").reset_index(drop = True)

    #Normalize_price
    #df_sales_data["price"] = (df_sales_data.price-df_sales_data.price.min())/(df_sales_data.price.max()-df_sales_data.price.min())
    df_sales_data["intercept"] =1

    
    
    #Randomly shuffle the data
    pvid_unique= df_sales_data.pvid.unique()
    random_perm = np.random.permutation(range(len(pvid_unique)))
    df_pv_id_rand = pd.DataFrame({"rand":random_perm, "pvid": pvid_unique})
    df_sales_data = df_sales_data.merge(df_pv_id_rand, on = "pvid", how = "inner").sort_values(by = "rand").reset_index(drop = True)
    
    #Split as test and train
    num_customers = df_sales_data.pvid.nunique()
    end_train = int(num_customers*percentage_train)*assort_size
    train_data = df_sales_data.loc[0:end_train-1,:]
    test_data = df_sales_data.loc[end_train:,:]
    
    return train_data, test_data





This is the code for the consideration sets MNL estimation

In [34]:
def Create_Sales_Data(df_sales_data, num_features, assort_size):
    
    """Set up the training data for Tensflow with clicks as consideration sets"""
    
    
    #only look at sales data with at least one click
    df_sales_data_w_click = df_sales_data.groupby(by = "pvid").filter(lambda x : x.is_click.sum()>0)
    
    
    #Get the offered products - all 6
    offered = np.concatenate(list(df_sales_data_w_click.groupby("pvid").\
                apply(lambda x:np.repeat(np.array(x.loc[:, feature_list])*np.repeat(np.array(x.loc[:, "is_click"]).reshape((assort_size,1)),\
                                        repeats=num_features, axis=1), repeats = max(x.is_buy.sum(),1), axis=0))))
    
    #Get the total number of assortment so we can reshape
    num_rows, num_cols = offered.shape
    total_assortments = int(num_rows/assort_size)
    
    #Reshape to get ready for tensorflow
    offered = offered.reshape((total_assortments,assort_size, num_features))
    
    #Get the vector of features for the purchased product
    purchased = np.concatenate(list(df_sales_data_w_click.groupby("pvid").\
            apply(lambda x: np.zeros(shape=(1,num_features)) if x.is_buy.sum()==0 else  np.array(x.loc[x.is_buy == 1, feature_list]) ).values))
    
    
    clicked = np.concatenate(list(df_sales_data_w_click.groupby("pvid").\
        apply(lambda x:np.repeat(np.array(x.loc[:, "is_click"]), repeats = max(x.is_buy.sum(),1), axis=0)))).reshape((total_assortments, assort_size))
    
    
    return offered, purchased, clicked

In [35]:
def Get_Click_Likelihood(df_sales_data):
    
    """Computes the click probs as empirical averages
    which is MLE estimates"""
    
    
    df_clicks = df_sales_data.loc[:,["is_click", "new_item_id", "item_id"]]
    df_clicks["num_offered"] = df_clicks.groupby(by = "new_item_id").is_click.transform(lambda x: len(x))
    df_clicks["num_clicked"] = df_clicks.groupby(by = "new_item_id").is_click.transform(lambda x: x.sum())
    
    df_clicks["Click_prob"] = df_clicks.num_clicked/df_clicks.num_offered
    
    neg_log_likelihood = -np.sum(np.log(1-df_clicks.loc[df_clicks.is_click == 0, "Click_prob"] )) - \
                            np.sum(np.log(df_clicks.loc[df_clicks.is_click == 1, "Click_prob"]))
    #This is enoug
    click_probs = df_clicks.loc[~df_clicks.duplicated(keep = "last", subset = ["item_id"]), ["item_id", "Click_prob"]]
    
    return neg_log_likelihood, click_probs

def Get_Click_Probs_Logistic_Regression(df_sales_data, feature_list, num_features):
    
    X = np.array(df_sales_data.loc[:, feature_list])
    y = np.array(df_sales_data.is_click)
    
    # instantiate a logistic regression model, and fit with X and y
    model = LogisticRegression()
    model = model.fit(X, y)
    
    return model.coef_[0].reshape((num_features,1)),model.intercept_[0]

def Get_Click_Probs_Cat_Boost(df_sales_data, feature_list, best_params):
    
    X = np.array(df_sales_data.loc[:, feature_list])
    y = np.array(df_sales_data.is_click)
    
    # instantiate a logistic regression model, and fit with X and y
#     params = best_params
#     cb = CatBoostClassifier(logging_level = "Silent")
#     model = GridSearchCV(cb, params, scoring="roc_auc", cv = 3).fit(X, y)
    
    model = CatBoostClassifier(logging_level = "Silent").fit(X, y)

    
    return model

def Get_Purchase_Probs_Cat_Boost(df_sales_data, feature_list, best_params):
    
    X = np.array(df_sales_data.loc[df_sales_data.is_click ==1, feature_list])
    y = np.array(df_sales_data.loc[df_sales_data.is_click ==1,"is_buy"])
    
    # instantiate a logistic regression model, and fit with X and y
    params = best_params
    cb = CatBoostClassifier(logging_level = "Silent")
    model = CatBoostClassifier(logging_level = "Silent").fit(X, y)
    
    #model = CatBoostClassifier(logging_level = "Silent").fit(X, y)

    
    return model




In [36]:
def Get_Click_Likelihood_Test(df_sales_data, click_probs):
    
    """Computes the test likelihood for the clicks
    """
    
    
    df_clicks = df_sales_data.loc[:,["is_click", "new_item_id", "item_id"]]
    df_clicks =  df_clicks.merge(click_probs, how = "left", on = "item_id")
    
    if df_clicks.Click_prob.isnull().sum()>0:
        #Replace the NAs with an average
        print("Missing clicks")
        df_clicks.fillna(value = {"Click_prob": df_clicks.Click_prob.mean()}, inplace = True)
        
    if df_clicks.loc[df_clicks.is_click == 1, "Click_prob"].min() == 0:
        print("click in test with 0 click prob")
        
    
    neg_log_likelihood = -np.sum(np.log(1-df_clicks.loc[df_clicks.is_click == 0, "Click_prob"] )) - \
                            np.sum(np.log(df_clicks.loc[df_clicks.is_click == 1, "Click_prob"]))
        

    
    return neg_log_likelihood



In [37]:


def Estimate_MNL_Clicks(train_data, num_features, assort_size, \
                        batch_size, learning_rate):

    offered_train, purchased_train, clicked_train = Create_Sales_Data(train_data, num_features, assort_size)


    #placeholders for offer data
    offer_mnl = tf.placeholder(tf.float32, [None, assort_size, num_features])

    #Placeholder for purchase data
    purchase_mnl = tf.placeholder(tf.float32, [None, num_features])

    #mask
    mask = tf.placeholder(tf.bool, [None, assort_size])

    #PLaceholder for mask
    zeros = tf.placeholder(tf.float32, [None, assort_size])


    #Create the variables to be estimated
    W = tf.Variable(tf.random_normal(shape=[num_features], mean=0, stddev=0.01), name="weights")


    #compute the log likelihood (negative because we minimize)
    first_term = tf.reduce_sum(tf.multiply(purchase_mnl, W),1)
    second_term =tf.log(tf.reduce_sum(tf.where(mask, tf.exp(tf.reduce_sum(tf.multiply(offer_mnl,W),2)), zeros), 1)  + 1) 
    cost = tf.reduce_sum(second_term-first_term)

    #optimization
    optimizer = tf.train.AdamOptimizer(learning_rate).minimize(cost)
    init = tf.global_variables_initializer()

    train_likelihood_list = []
    epoch_count= 0
    percent_improvement=100
    with tf.Session() as sess:
        sess.run(init)
        while epoch_count<3000 and percent_improvement> 0.00001:

            num_batches = int(purchased_train.shape[0]/batch_size)
            log_like = 0
            for b in range(num_batches+1):
                if b<num_batches:
                    purchase_batch = purchased_train[b*batch_size:(b+1)*batch_size,:]
                    offer_batch = offered_train[b*batch_size:(b+1)*batch_size,:,:]
                    click_batch = clicked_train[b*batch_size:(b+1)*batch_size,:]
                else:


                    purchase_batch = purchased_train[b*batch_size:,:]
                    offer_batch = offered_train[b*batch_size:,:,:]
                    click_batch = clicked_train[b*batch_size:,:]

                num_rows = purchase_batch.shape[0] 

                _,neg_log_like = sess.run(fetches=[optimizer, cost], feed_dict={offer_mnl:offer_batch, purchase_mnl:purchase_batch,\
                                                                   mask:click_batch, zeros:np.zeros((num_rows, assort_size))})
                log_like+=neg_log_like
                
            epoch_count+=1
            train_likelihood_list.append(log_like)
            if len(train_likelihood_list) >=100 and epoch_count%10==0:
                last_ten = min(train_likelihood_list[-10:])
                other_min  = min(train_likelihood_list[:-10])
                percent_improvement = (abs(last_ten - other_min)/other_min)
                #print(percent_improvement)
#             print("Epoch:%d" %epoch_count)
#             print(log_like)
        
            if epoch_count==1000:
                print(percent_improvement)
        
        print(percent_improvement)
        W_final_clicks = sess.run(W)
        
        return W_final_clicks


$\textbf{________________________________________________________________________________________________________________________________________________________________________}$

Here is where we are fitting the classic MNL, where each offerset has size 6.

In [38]:
def Create_Sales_Data_Classic(df_sales_data, num_features, assort_size):
    
    """Set up the training data for Tensflow with full offer sets"""
    
    
    
    #Get the offered products - all 6
    offered = np.concatenate(list(df_sales_data.groupby("pvid").\
                apply(lambda x:np.repeat(np.array(x.loc[:, feature_list]), repeats = max(x.is_buy.sum(),1), axis=0))))
    
    
    #Get the total number of assortment so we can reshape
    num_rows, num_cols = offered.shape
    total_assortments = int(num_rows/assort_size)
    
    #Reshape to get ready for tensorflow
    offered = offered.reshape((total_assortments,assort_size, num_features))
    
    #Get the vector of features for the purchased product
    purchased = np.concatenate(list(df_sales_data.groupby("pvid").\
            apply(lambda x: np.zeros(shape=(1,num_features)) if x.is_buy.sum()==0 else  np.array(x.loc[x.is_buy == 1, feature_list]) ).values))
    
    

    
    return offered, purchased


In [39]:
#Begin setting up computation graph for 


def Estimate_MNL_Classic(train_data, num_features, assort_size, \
                        batch_size, learning_rate):

    offered_classic_train, purchased_classic_train = Create_Sales_Data_Classic(train_data, num_features, assort_size)

    #placeholders for offer data
    offer_mnl = tf.placeholder(tf.float32, [None, assort_size, num_features])

    #Placeholder for purchase data
    purchase_mnl = tf.placeholder(tf.float32, [None, num_features])

    #Create the variables to be estimated
    W = tf.Variable(tf.random_normal(shape=[num_features], mean=0, stddev=0.01), name="weights")

    #compute the log likelihood (negative because we minimize)
    first_term = tf.reduce_sum(tf.multiply(purchase_mnl, W),1)
    second_term =tf.log(tf.reduce_sum( tf.exp(tf.reduce_sum(tf.multiply(offer_mnl,W),2)), 1)  + 1) 
    cost = tf.reduce_sum(second_term-first_term)

    #optimization
    optimizer = tf.train.AdamOptimizer(learning_rate).minimize(cost)
    init = tf.global_variables_initializer()

    train_likelihood_list = []
    epoch_count= 0
    percent_improvement=100
    with tf.Session() as sess:
        sess.run(init)
        while epoch_count<3000 and percent_improvement> 0.00001:
            num_batches = int(purchased_classic_train.shape[0]/batch_size)
            log_like = 0
            for b in range(num_batches+1):
                if b<num_batches:
                    purchase_batch = purchased_classic_train[b*batch_size:(b+1)*batch_size,:]
                    offer_batch = offered_classic_train[b*batch_size:(b+1)*batch_size,:,:]
                else:


                    purchase_batch = purchased_classic_train[b*batch_size:,:]
                    offer_batch = offered_classic_train[b*batch_size:,:,:]

                num_rows = purchase_batch.shape[0] 

                _,neg_log_like = sess.run(fetches=[optimizer, cost], \
                                          feed_dict={offer_mnl:offer_batch, purchase_mnl:purchase_batch})
                log_like+=neg_log_like

            epoch_count+=1
            train_likelihood_list.append(log_like)
            if len(train_likelihood_list) >=100 and epoch_count%10==0:
                last_ten = min(train_likelihood_list[-10:])
                other_min  = min(train_likelihood_list[:-10])
                percent_improvement = (abs(last_ten - other_min)/other_min)
                #print(percent_improvement)

#             print("Epoch:%d" %epoch_count)
#             print(log_like)
            
            if epoch_count==1000:
                print(percent_improvement)
                
        print(percent_improvement)
        W_final_classic = sess.run(W)
        
        return W_final_classic

    

Below are function to do the testing.

In [40]:
def MNL_Log_Like_Classic(offered_classic_test, purchased_classic_test, W_classic):
    """Computes class MNL log likelihood"""
    
    
    #num_assortments
    T = purchased_classic_test.shape[0]
    log_like = 0
    for t in range(T):
        
        numerator = np.exp(np.dot(purchased_classic_test[t,:], W_classic))
        denom = np.sum(np.exp(np.matmul(offered_classic_test[t,:,:],W_classic))) +1
        log_like+=np.log(numerator/denom)
        
        
    return log_like

def MNL_Log_Like_Clicks(df_data, click_probs, W, offered_test, purchased_test):
    """Likelihood with clicks"""
    
    T = purchased_test.shape[0]
    click_log_like = Get_Click_Likelihood_Test(df_data, click_probs)
    log_like = click_log_like
    for t in range(T):
        numerator = np.exp(np.dot(purchased_test[t,:], W))
        denom_vals = np.exp(np.matmul(offered_test[t,:,:],W))
        denom = 1
        for probit in denom_vals:
            if probit!=1:
                denom+=probit
        log_like-=np.log(numerator/denom)
    
    
    return log_like

def Click_Test_Like(df_test,cat_model):
    
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    

    
    #Add click_probs LR
    df_purchase["cat_click"] = cat_model.predict_proba(np.array(df_purchase.loc[:, feature_list]))[:,1]
    
    LL = sum(df_purchase.apply(lambda x: math.log(x.cat_click) if x.is_click ==1 else math.log(1 - x.cat_click), axis=1 ))
    
    return LL/df_purchase.pvid.nunique()
    
    
    
        

In [41]:
def Get_Top_Choice_Prob_Classic(df_test, W_final_classic ):
    
    """For customers who made a purchase, looks at if the predicted one was purchased 
    and the average rank of the predicted option"""
    
    #Only look at customers who made exactly one purchase
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    
    #Add probit term for each product
    df_purchase["probit"]= df_purchase.apply(lambda x: math.exp(sum(x[feature_list]*W_final_classic)), axis = 1 ) 
    
    #Sort the probits by group
    df_purchase.sort_values(by = ["pvid", "probit"], ascending = [False, False], inplace = True)
    df_purchase.reset_index(drop = True, inplace = True)
    
    #Get the rank of each offered product
    df_purchase["rank"] = df_purchase.index%6 +1
    avg_rank = df_purchase.loc[df_purchase.is_buy == 1, "rank"].mean()
    frac_correct = df_purchase.loc[df_purchase.is_buy == 1, "rank"].value_counts(normalize = True)[1]
    
    #Compute log-like
    df_purchase["mnl_denom"] = df_purchase.groupby("pvid").probit.transform(lambda x: 1 + sum(x))
    
    
    df_purchase["log_like"] = df_purchase.apply(lambda x: math.log(x.probit/x.mnl_denom), axis = 1 )  
    
    log_like = df_purchase.loc[df_purchase.is_buy == 1,:].log_like.sum()
    
    
    return avg_rank, frac_correct, log_like

    

In [47]:
def Powerset(iterable):
    s = list(iterable)
    return [list(a) for r in range(len(s)+1) for a in combinations(s, r) ]


def Get_PP_clicks(df_assort, assort_size):
    
    """Computes purchase probs for the Click model"""
    
    df_assort.reset_index(drop = True, inplace = True)
    all_assortments = Powerset(list(range(assort_size)))
    

    
    bought = []
    pp_list = []
    for i in range(assort_size):
        numerator = df_assort.loc[i,"pref_weight"]
        bought.append(df_assort.loc[i,"is_buy"])
        pp=0
        for assort in all_assortments:
            if i in assort:
                total_click_prob = 1
                total_sum_weights = 1
                for prod in range(assort_size):
                    if prod not in assort:
                        total_click_prob*=(1-df_assort.loc[prod,"Click_prob"])
                    else:
                        total_click_prob*=(df_assort.loc[prod,"Click_prob"])
                        total_sum_weights+=df_assort.loc[prod,"pref_weight"]
                pp+=(numerator/total_sum_weights)*total_click_prob
        pp_list.append(pp)
                        
            
    return pd.DataFrame({"purchase_prob":pp_list, "bought": bought})


def Get_PP_clicks_CTR(df_assort, assort_size):
    
    """Computes purchase probs for the Click model"""
    
    df_assort.reset_index(drop = True, inplace = True)
    all_assortments = Powerset(list(range(assort_size)))
    
    
    bought = []
    pp_list = []
    for i in range(assort_size):
        numerator = df_assort.loc[i,"pref_weight"]
        bought.append(df_assort.loc[i,"is_buy"])
        pp=0
        for assort in all_assortments:
            if i in assort:
                total_click_prob = 1
                total_sum_weights = 1
                for prod in range(assort_size):
                    if prod not in assort:
                        total_click_prob*=(1-df_assort.loc[prod,"i_ctr"])
                    else:
                        total_click_prob*=(df_assort.loc[prod,"i_ctr"])
                        total_sum_weights+=df_assort.loc[prod,"pref_weight"]
                pp+=(numerator/total_sum_weights)*total_click_prob
        pp_list.append(pp)
                        
            
    return pd.DataFrame({"purchase_prob":pp_list, "bought": bought})

def Get_PP_clicks_LR(df_assort, assort_size):
    
    """Computes purchase probs for the Click model"""
    
    df_assort.reset_index(drop = True, inplace = True)
    all_assortments = Powerset(list(range(assort_size)))
    
    
    bought = []
    pp_list = []
    for i in range(assort_size):
        numerator = df_assort.loc[i,"pref_weight"]
        bought.append(df_assort.loc[i,"is_buy"])
        pp=0
        for assort in all_assortments:
            if i in assort:
                total_click_prob = 1
                total_sum_weights = 1
                for prod in range(assort_size):
                    if prod not in assort:
                        total_click_prob*=(1-df_assort.loc[prod,"ctr_lr"])
                    else:
                        total_click_prob*=(df_assort.loc[prod,"ctr_lr"])
                        total_sum_weights+=df_assort.loc[prod,"pref_weight"]
                pp+=(numerator/total_sum_weights)*total_click_prob
        pp_list.append(pp)
                        
            
    return pd.DataFrame({"purchase_prob":pp_list, "bought": bought})

def Get_PP_clicks_Cat(df_assort, assort_size):
    
    """Computes purchase probs for the Click model"""
    
    df_assort.reset_index(drop = True, inplace = True)
    all_assortments = Powerset(list(range(assort_size)))
    
    
    bought = []
    pp_list = []
    prod_id = []
    for i in range(assort_size):
        numerator = df_assort.loc[i,"w_clicks"]
        prod_id.append(df_assort.loc[i,"item_id"])
        bought.append(0)
        pp=0
        for assort in all_assortments:
            if i in assort:
                total_click_prob = 1
                total_sum_weights = 0
                for prod in range(assort_size):
                    if prod not in assort:
                        total_click_prob*=(1-df_assort.loc[prod,"ctr_cat"])
                    else:
                        total_click_prob*=(df_assort.loc[prod,"ctr_cat"])
                        total_sum_weights+=df_assort.loc[prod,"w_clicks"]
                pp+=(numerator/total_sum_weights)*total_click_prob
        pp_list.append(pp)
        
    
    elements = list(range(assort_size))
    probs = [p/sum(pp_list) for p in pp_list]
    purchased = np.random.choice(elements, 1, p=probs)[0]
    bought[purchased]=1
                        
            
    return pd.DataFrame({"is_buy": bought, "item_id":prod_id})


In [43]:

def Get_Top_Choice_Prob_Classic_Clicks(df_test, click_probs, W, assort_size, num_features):
    
    """For customers who made a purchase, looks at if the predicted one was purchased 
    and the average rank of the predicted option"""
    
    #Only look at customers who made exactly one purchase
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    
    #Add the click probs 
    df_purchase_clicks =  df_purchase.merge(click_probs, how = "inner", on = "item_id")
    
    #Add the preference weights
    df_purchase_clicks["pref_weight"] = np.exp(np.matrix(df_purchase_clicks.loc[:,feature_list])*W.reshape((num_features,1)))
    
    df_purchase_clicks= df_purchase_clicks.groupby("pvid").filter(lambda x: len(x) == 6)

    
    
    #Add the purchase probability
    click_probs  = df_purchase_clicks.groupby("pvid").apply(Get_PP_clicks, assort_size = assort_size).reset_index()
    
    del click_probs["level_1"]


    #Sort the purchase probs by group
    click_probs.sort_values(by = ["pvid", "purchase_prob"], ascending = [False, False], inplace = True)
    click_probs.reset_index(drop = True, inplace = True)
    
    #Get the rank of each offered product
    click_probs["rank"] = click_probs.index%6 +1
    avg_rank = click_probs.loc[click_probs.bought == 1, "rank"].mean()
    frac_correct = click_probs.loc[click_probs.bought == 1, "rank"].value_counts(normalize = True)[1]
    
    
    return avg_rank, frac_correct
    

In [44]:
def Get_Top_Choice_Prob_Classic_CTR_Clicks(df_test, W, assort_size, num_features):
    
    """For customers who made a purchase, looks at if the predicted one was purchased 
    and the average rank of the predicted option"""
    
    #Only look at customers who made exactly one purchase
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    #print("Number of data points:", df_purchase.pvid.nunique())

    
    #Add the preference weights
    df_purchase["pref_weight"] = np.exp(np.matrix(df_purchase.loc[:,feature_list])*W.reshape((num_features,1)))
    
    
    #Add the purchase probability
    click_probs  =  df_purchase.groupby("pvid").apply(Get_PP_clicks_CTR, assort_size = assort_size).reset_index()
    
    del click_probs["level_1"]


    #Sort the purchase probs by group
    click_probs.sort_values(by = ["pvid", "purchase_prob"], ascending = [False, False], inplace = True)
    click_probs.reset_index(drop = True, inplace = True)
    
    #Get the rank of each offered product
    click_probs["rank"] = click_probs.index%6 +1
    avg_rank = click_probs.loc[click_probs.bought == 1, "rank"].mean()
    frac_correct = click_probs.loc[click_probs.bought == 1, "rank"].value_counts(normalize = True)[1]
    
    
    return avg_rank, frac_correct

def Get_Top_Choice_Prob_Classic_LR_Clicks(df_test, W, W_clicks ,intercept, assort_size, num_features):
    
    """For customers who made a purchase, looks at if the predicted one was purchased 
    and the average rank of the predicted option"""
    
    #Only look at customers who made exactly one purchase
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    #print("Number of data points:", df_purchase.pvid.nunique())

    
    #Add the preference weights
    df_purchase["pref_weight"] = np.exp(np.matrix(df_purchase.loc[:,feature_list])*W.reshape((num_features,1)))
    
    #Add click_probs LR
    df_purchase["ctr_lr"] = expit(intercept + np.matmul(np.array(df_purchase.loc[:, feature_list]),W_clicks))
    
    
    #Add the purchase probability
    click_probs  =  df_purchase.groupby("pvid").apply(Get_PP_clicks_LR, assort_size = assort_size).reset_index()
    
    del click_probs["level_1"]


    #Sort the purchase probs by group
    click_probs.sort_values(by = ["pvid", "purchase_prob"], ascending = [False, False], inplace = True)
    click_probs.reset_index(drop = True, inplace = True)
    
    #Get the rank of each offered product
    click_probs["rank"] = click_probs.index%6 +1
    avg_rank = click_probs.loc[click_probs.bought == 1, "rank"].mean()
    frac_correct = click_probs.loc[click_probs.bought == 1, "rank"].value_counts(normalize = True)[1]
    
    
    return avg_rank, frac_correct
    
    
def Get_Top_Choice_Prob_Classic_Cat_Boost_Clicks(df_test, W, cat_model, assort_size, feature_list, num_features):
    
    """For customers who made a purchase, looks at if the predicted one was purchased 
    and the average rank of the predicted option"""
    
    #Only look at customers who made exactly one purchase
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    #print("Number of data points:", df_purchase.pvid.nunique())

    
    #Add the preference weights
    df_purchase["pref_weight"] = np.exp(np.matrix(df_purchase.loc[:,feature_list])*W.reshape((num_features,1)))
    
    #Add click_probs LR
    df_purchase["ctr_lr"] = cat_model.predict_proba(np.array(df_purchase.loc[:, feature_list]))[:,1]
    
    
    #Add the purchase probability
    click_probs  =  df_purchase.groupby("pvid").apply(Get_PP_clicks_LR, assort_size = assort_size).reset_index()
    
    
    
    del click_probs["level_1"]


    #Sort the purchase probs by group
    click_probs.sort_values(by = ["pvid", "purchase_prob"], ascending = [False, False], inplace = True)
    click_probs.reset_index(drop = True, inplace = True)
    
    #Get the rank of each offered product
    click_probs["rank"] = click_probs.index%6 +1
    avg_rank = click_probs.loc[click_probs.bought == 1, "rank"].mean()
    frac_correct = click_probs.loc[click_probs.bought == 1, "rank"].value_counts(normalize = True)[1]
    
    click_probs["log_like"] = click_probs.apply(lambda x: math.log(x.purchase_prob), axis = 1 ) 
    
    log_like = click_probs.loc[click_probs.bought == 1,:].log_like.sum()
    
    
    
    
    return avg_rank, frac_correct, log_like

def Get_Top_Choice_Prob_ML_Cat_Boost_Clicks(df_test,cat_model_buy, cat_model, assort_size, feature_list, num_features):
    
    #Only look at customers who made exactly one purchase
    df_purchase = df_test.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    #print("Number of data points:", df_purchase.pvid.nunique())

    
    #Add the preference weights
    df_purchase["pp"] = cat_model_buy.predict_proba(np.array(df_purchase.loc[:, feature_list]))[:,1]
    
    #Add click_probs LR
    df_purchase["ctr_lr"] = cat_model.predict_proba(np.array(df_purchase.loc[:, feature_list]))[:,1]
    
    
    
    #Add the purchase probability
    df_purchase["purchase_prob"]  =  df_purchase["pp"]* df_purchase["ctr_lr"]
    


    #Sort the purchase probs by group
    df_purchase.sort_values(by = ["pvid", "purchase_prob"], ascending = [False, False], inplace = True)
    df_purchase.reset_index(drop = True, inplace = True)
    
    #Get the rank of each offered product
    df_purchase["rank"] = df_purchase.index%6 +1
    avg_rank = df_purchase.loc[df_purchase.is_buy == 1, "rank"].mean()
    frac_correct = df_purchase.loc[df_purchase.is_buy == 1, "rank"].value_counts(normalize = True)[1]
    
    
    return avg_rank, frac_correct
    
    
    

In [45]:
def Get_Features(list_prods, feature_list, sales_data):
    
    
    
    feature_dict = {}
    
    for prod in list_prods:
        
        prod_data = sales_data.loc[sales_data.item_id == prod, feature_list].reset_index(drop = True)
        rand_row = np.random.randint(prod_data.shape[0])
        features = list(prod_data.loc[rand_row,:])
        feature_dict[prod] = features
        
        
    return feature_dict


$\textbf{________________________________________________________________________________________________________________________________________________________________________}$


In [48]:


#Get file path and feature list
data_files = [file_ for file_ in os.listdir("Data/") if ".csv" in file_]
num_sellers = len(data_files)
#data_files = ["b2436f18c140cf163e28ae62f733aaab9624ed766185f1c2f27ce5c91c76ecff-output_data.csv"]
assort_size = 6
feature_list = ["f%d" %i for i in range(1,24) ]+ ["i_cvr" , "intercept", "price"]
num_features = len(feature_list)
assortment_sellers = ["ef81bdd8e8dab862e96443a458f19b2e9467006fe892e493a72d96900882093a", "c4e6adee276fa74dcc29c60c696de4836d8d9014200040a45ba06a96df08395d", \
                       "b85461bc1516523e40419912ad9df6d9d8935df96cc911a8a84b263bf23a5830"]
#Fraction of data used for training
percentage_train = 1

#Fraction of most popuar products to include
frac_include = 1

#parameters for gradient descent
batch_size = 20_000
learning_rate = 0.03
num_trials = 1

#Create results files
f = open("Data/Results/parameters.csv", "w")
f.write("Seller, Train_perc, frac_prods, num_prods\n")
g_classic = open("Data/Results/results_classic.csv", "w")
g_clicks = open("Data/Results/results_clicks.csv", "w")
g_ctr = open("Data/Results/results_ml.csv", "w")
g_cat = open("Data/Results/results_cat.csv", "w")
g_classic.write("Seller, Trial, Train_perc, frac_prods, rank, accuracy, log_like\n")
g_clicks.write("Seller, Trial, Train_perc, frac_prods, rank, accuracy\n")
g_ctr.write("Seller, Trial, Train_perc, frac_prods, rank, accuracy\n")
g_cat.write("Seller, Trial, Train_perc, frac_prods, rank, accuracy, log_like\n")
f.close()

mle = open("Data/Results/click_mle.csv", "w")


for t in range(num_trials):
    print(t)
    best_params = {'depth': [3,4,5]}
    train_dict = {}
    test_dict = {}
    for seller in data_files:
        
        #TRAIN/TEST
        file_path = "Data/" + seller
        train_data_seller, test_data_seller = Get_Test_Train(seller, file_path, percentage_train, frac_include)
#         if len(train_dict)>0:
#             #Because pvids for each seller start from 0
#             train_data_seller.new_pvid = train_data_seller.new_pvid + max(train_dict[s-1].new_pvid.max(), test_dict[s-1].new_pvid.max()) + 1            

        train_dict[seller] = train_data_seller
        test_dict[seller] = test_data_seller

            
    train_data = pd.concat([train_dict[seller]  for seller in data_files])
    


    #Estimation with clicks
    W_final_clicks = Estimate_MNL_Clicks(train_data, num_features, assort_size, \
                            batch_size, learning_rate)
    
    #Cat boost clicks
    cat_model =  Get_Click_Probs_Cat_Boost(train_data, feature_list, best_params)
    


    #Generate new sales data
    train_data_mnl = train_data.copy()
    
    df_purchase = train_data_mnl.groupby("pvid").filter(lambda x: x.is_buy.sum()==1).reset_index(drop = True)
    
    df_no_purchase = train_data_mnl.groupby("pvid").filter(lambda x: x.is_buy.sum()==0).reset_index(drop = True)

    del df_purchase["is_buy"]

    #Add the preference weights
    df_purchase["w_clicks"] = np.exp(np.matrix(df_purchase.loc[:,feature_list])*W_final_clicks.reshape((num_features,1)))

    #Add click_probs LR
    df_purchase["ctr_cat"] = cat_model.predict_proba(np.array(df_purchase.loc[:, feature_list]))[:,1]

    #Add the purchase probability
    click_data  =  df_purchase.groupby("pvid").apply(Get_PP_clicks_Cat, assort_size = assort_size).reset_index()
    
    print("Done getting new sales data")

    del click_data["level_1"]

    df_merged = df_purchase.merge(click_data, on = ["pvid", "item_id"], how= "inner")
    df_merged_final = pd.concat([df_merged.loc[:, train_data.columns], df_no_purchase] )
    
    
    #Classic MNL Estimation
    W_final_classic = Estimate_MNL_Classic(df_merged_final, num_features, assort_size, \
                            batch_size, 0.05)
    

    if t==num_trials-1:
        for t_ in range(100):
            print(t_)
            for seller in assortment_sellers:
                #Create results files
                f = open("assortment_cases/%s_%d.txt" %(seller,t_), "w")
                seller_name = seller + "-output_data.csv"

                df_full = train_dict[seller_name] 

                #Get the number of products
                list_prods = list(df_full.item_id.unique())
                num_prods = len(list_prods)


                f.write("%d\n" %num_prods)

                #For each product get features
                feature_dict = Get_Features(list_prods, feature_list, df_full)


                #Write the feature vals and revenues
                for prod in feature_dict:

                    features = feature_dict[prod]
                    price = feature_dict[prod][-1]
                    pref_weight = math.exp(np.dot(features, W_final_clicks))
                    pref_weight_mnl = math.exp(np.dot(features, W_final_classic))
                    click_prob = cat_model.predict_proba(np.array(features).reshape((1, num_features)))[0,1]
                    #purchase_prob_ml = cat_model_buy.predict_proba(np.array(features).reshape((1, num_features)))[0,1]
                    true_pp =0.1
                    f.write("%s,%0.4f,%0.4f,%0.4f,%0.2f, %0.4f\n" %(prod, pref_weight, pref_weight_mnl, click_prob, price, true_pp))

                f.close()
    
    



0


/Users/feldman/anaconda/lib/python3.6/site-packages/numpy/lib/arraysetops.py:522: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  mask |= (ar1 == a)


0.0008288303768487787
0.0007784722053854796
Done getting new sales data
7.657015100727887e-06
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
